# ChessGPT on Colab
Train a causal transformer from scratch on chess move histories, then compare human imitation, legality supervision and engine preferences.

In Colab, select **Runtime → Change runtime type → GPU**, then run the cells in order. Allow the Drive mount so a new session can resume. The default training budgets total 4.5 hours. Downloading games, parsing PGN, CPU engine labeling and evaluation take additional time. Each phase has its own run cell. You can stop after the baseline and return later.

For a quick pipeline check, choose a separate run directory and reduce the budgets and sample counts in the configuration cell. A short run is a wiring check. Keep the original run directory and settings when resuming a full experiment.

## 1. Install the rules library and CPU chess engine

In [ ]:
%pip -q install chess==1.11.2 zstandard
import subprocess
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "stockfish"], check=True)
from google.colab import drive
drive.mount("/content/drive")

## 2. Configure the experiment
Use a new directory for each architecture, batch size or dataset. A smaller batch uses less GPU memory. To use your own data, set SOURCE to the path of an uncompressed standard PGN on Drive. The default archive streams until MAX_GAMES distinct games have been read. Drive stores the parsed games, checkpoints, labels and evaluation reports.

In [ ]:
from pathlib import Path
import hashlib, io, json, math, os, random, time, uuid
from collections import defaultdict
from contextlib import nullcontext
import requests, zstandard
import chess, chess.pgn, chess.engine
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path("/content/drive/MyDrive/chessgpt/run01")
SOURCE = "https://database.lichess.org/standard/lichess_db_standard_rated_2015-01.pgn.zst"
MAX_GAMES, SEED, CTX = 100000, 42, 256
WIDTH, HEADS, LAYERS, BATCH = 512, 8, 8, 16
BASE_SECONDS, BRANCH_SECONDS = 10800, 1800
LABEL_COUNT, EVAL_COUNT, OPENINGS = 8192, 300, 100
LABEL_NODES, TEST_NODES = 512, 4096
ENGINE = "/usr/games/stockfish"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT.mkdir(parents=True, exist_ok=True)
print("Training device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE)

def atomic(path, writer):
    temp = path.with_name(path.name + ".tmp")
    writer(temp)
    os.replace(temp, path)

def save_json(path, value):
    atomic(path, lambda p: p.write_text(json.dumps(value)))

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

# Include every queen-ray, knight and promotion move, including underpromotions.
moves = set()
for start in chess.SQUARES:
    for end in chess.SQUARES:
        dx = abs(chess.square_file(start) - chess.square_file(end))
        dy = abs(chess.square_rank(start) - chess.square_rank(end))
        if start != end and (dx == dy or min(dx, dy) == 0 or sorted((dx, dy)) == [1, 2]):
            move = chess.Move(start, end).uci()
            moves.add(move)
            if dx <= 1 and (chess.square_rank(start), chess.square_rank(end)) in [(1, 0), (6, 7)]:
                moves.update(move + piece for piece in "qrbn")
VOCAB = sorted(moves) + ["<start>", "<pad>"]
ids = {move: i for i, move in enumerate(VOCAB)}
BOS, PAD = ids["<start>"], ids["<pad>"]

def replay(prefix):
    board = chess.Board()
    for idx in prefix:
        board.push_uci(VOCAB[idx])
    return board

def legal_ids(board):
    return [ids[m.uci()] for m in board.legal_moves]

settings = dict(source=SOURCE,max_games=MAX_GAMES,seed=SEED,context=CTX,width=WIDTH,heads=HEADS,
                layers=LAYERS,batch=BATCH,base_seconds=BASE_SECONDS,branch_seconds=BRANCH_SECONDS,
                label_count=LABEL_COUNT,eval_count=EVAL_COUNT,openings=OPENINGS,label_nodes=LABEL_NODES,
                test_nodes=TEST_NODES,torch_version=torch.__version__,vocabulary=VOCAB)
save_json(ROOT/("settings-"+fingerprint(settings)[:12]+".json"),settings)

## 3. Prepare the dataset and freeze evaluation histories

In [ ]:
def read_games(stream):
    games = defaultdict(list)
    seen = set()
    truncated = 0
    while len(seen) < MAX_GAMES:
        game = chess.pgn.read_game(stream)
        if game is None:
            break
        if game.errors or game.headers.get("Variant", "Standard") != "Standard" or game.board().fen() != chess.STARTING_FEN:
            continue
        sequence = [m.uci() for m in game.mainline_moves()]
        if not sequence:
            continue
        # Deduplicate the visible prefix too, including games truncated to context.
        digest = fingerprint([SEED, sequence[:CTX]])
        if digest in seen:
            continue
        seen.add(digest)
        bucket = int(digest[:8], 16) % 100
        split = "test" if bucket == 0 else "valid" if bucket == 1 else "train"
        games[split].append([ids[m] for m in sequence[:CTX]])
        truncated += len(sequence) > CTX
    if any(not games[s] for s in ("train", "valid", "test")):
        raise ValueError("Need more distinct games to populate every split.")
    return dict(games), truncated

def prepare():
    path = ROOT / "games.pt"
    signature = fingerprint([SOURCE, MAX_GAMES, SEED, CTX, VOCAB])
    if path.exists():
        artifact = torch.load(path, weights_only=False)
        if artifact["signature"] != signature:
            raise ValueError("Choose a new ROOT for the changed dataset.")
        return artifact
    if SOURCE.startswith("https://"):
        with requests.get(SOURCE, stream=True, timeout=(15, 120)) as response:
            response.raise_for_status()
            with zstandard.ZstdDecompressor().stream_reader(response.raw) as raw:
                games, truncated = read_games(io.TextIOWrapper(raw))
    else:
        with open(SOURCE, encoding="utf-8") as stream:
            games, truncated = read_games(stream)
    artifact = dict(games=games, truncated=truncated, signature=signature, source=SOURCE)
    atomic(path, lambda p: torch.save(artifact, p))
    return artifact

data = prepare()
games = data["games"]
print({s: len(g) for s,g in games.items()}, "truncated games:", data["truncated"])

def sample_rows(split, count):
    rng = random.Random(SEED + ["train","valid","test"].index(split))
    rows = []
    for lo, hi in [(0, 19), (20, 59), (60, CTX-1)]:
        eligible = [i for i,g in enumerate(games[split]) if len(g) > lo]
        if eligible:
            for _ in range((count + 2)//3):
                i = rng.choice(eligible)
                rows.append((i, rng.randint(lo, min(hi, len(games[split][i])-1))))
    rng.shuffle(rows)
    return list(dict.fromkeys(rows))[:count]

frozen = {s: sample_rows(s, EVAL_COUNT) for s in ("test","valid")}
opening_pool = [(i, min(12, len(g)//2)) for i,g in enumerate(games["test"]) if len(g) >= 8]
openings = random.Random(SEED).sample(opening_pool, min(OPENINGS, len(opening_pool)))
save_json(ROOT/"evaluation-manifest.json", dict(positions=frozen, openings=openings))

## 4. Define the model and training loop

In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(WIDTH), nn.LayerNorm(WIDTH)
        self.qkv = nn.Linear(WIDTH, WIDTH*3)
        self.out = nn.Linear(WIDTH, WIDTH)
        self.mlp = nn.Sequential(nn.Linear(WIDTH, WIDTH*4), nn.GELU(), nn.Linear(WIDTH*4, WIDTH))

    def forward(self, x):
        b,t,d = x.shape
        q,k,v = self.qkv(self.norm1(x)).reshape(b,t,3,HEADS,d//HEADS).permute(2,0,3,1,4).unbind()
        attn = F.scaled_dot_product_attention(q,k,v,is_causal=True)
        x = x + self.out(attn.transpose(1,2).reshape(b,t,d))
        return x + self.mlp(self.norm2(x))

class ChessGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(len(VOCAB), WIDTH)
        self.position = nn.Embedding(CTX, WIDTH)
        self.blocks = nn.Sequential(*(Block() for _ in range(LAYERS)), nn.LayerNorm(WIDTH))
        nn.init.normal_(self.tokens.weight, std=.02)
        nn.init.normal_(self.position.weight, std=.02)

    def forward(self, tokens):
        x = self.tokens(tokens) + self.position(torch.arange(tokens.size(1), device=tokens.device))
        return F.linear(self.blocks(x), self.tokens.weight)

def padded(rows):
    tensors = [torch.tensor(row, dtype=torch.long) for row in rows]
    return nn.utils.rnn.pad_sequence(tensors, batch_first=True, padding_value=PAD).to(DEVICE)

@torch.inference_mode()
def last_logits(model, prefixes):
    x = padded([[BOS]+p for p in prefixes])
    return model(x)[torch.arange(len(prefixes), device=DEVICE), x.ne(PAD).sum(1)-1].float()

def checkpoint(name):
    return torch.load(ROOT/(name+".pt"), map_location="cpu", weights_only=False)

def load_model(name):
    state = checkpoint(name)
    model = ChessGPT().to(DEVICE)
    model.load_state_dict(state["model"])
    return model.eval()

def train(name, budget, labels=None):
    base = name == "base"
    lr = 3e-4 if base else 5e-5
    path = ROOT/(name+".pt")
    parent = None if base else checkpoint("base")
    signature = fingerprint([data["signature"], WIDTH, HEADS, LAYERS, BATCH, lr,
                             None if base else [parent["id"],parent["step"],fingerprint(labels)]])
    torch.manual_seed(SEED)
    rng = random.Random(SEED)
    model = ChessGPT().to(DEVICE)
    if parent:
        model.load_state_dict(parent["model"])
        del parent
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE=="cuda")
    step, tokens, elapsed, run_id = 0, 0, 0., uuid.uuid4().hex
    if path.exists():
        state = checkpoint(name)
        if state["signature"] != signature:
            raise ValueError("Incompatible checkpoint. Select a fresh ROOT.")
        model.load_state_dict(state["model"])
        opt.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        rng.setstate(state["rng"])
        step,tokens,elapsed,run_id = (state[k] for k in ("step","tokens","elapsed","id"))
        del state
    print(name, sum(p.numel() for p in model.parameters()), "parameters; starting at", step)
    if elapsed >= budget:
        return
    def save():
        payload = dict(model=model.state_dict(), optimizer=opt.state_dict(), scaler=scaler.state_dict(),
                       step=step,tokens=tokens,elapsed=elapsed,rng=rng.getstate(),id=run_id,signature=signature)
        atomic(path, lambda p: torch.save(payload,p))
    last_save = elapsed
    start = time.monotonic()
    initial_time = elapsed
    warm_step = step+5
    while elapsed < budget:
        opt.zero_grad(set_to_none=True)
        if base:
            batch = rng.choices(games["train"], k=BATCH)
            x, y = padded([[BOS]+g[:-1] for g in batch]), padded(batch)
            count = sum(map(len,batch))
        else:
            rows = rng.choices(labels,k=BATCH)
            prefixes = [games["train"][r[0]][:r[1]] for r in rows]
            x = padded([[BOS]+p for p in prefixes])
            y = torch.tensor([games["train"][r[0]][r[1]] for r in rows],device=DEVICE)
            count = len(rows)
        with torch.autocast("cuda",dtype=torch.float16) if DEVICE=="cuda" else nullcontext():
            logits = model(x)
            if not base:
                logits = logits[torch.arange(len(rows),device=DEVICE),x.ne(PAD).sum(1)-1].float()
            loss = F.cross_entropy(logits.reshape(-1,len(VOCAB)), y.flatten(), ignore_index=PAD)
            if name in ("B","C"):
                legal = padded([r[2] for r in rows])
                legal_logits = logits.gather(1,legal).masked_fill(legal.eq(PAD),-torch.inf)
                loss = loss + (logits.logsumexp(1)-legal_logits.logsumexp(1)).mean()
            if name=="C":
                pairs = [(i,*r[3]) for i,r in enumerate(rows) if r[3]]
                if pairs:
                    ix,win,lose = torch.tensor(pairs,device=DEVICE).T
                    loss = loss + F.softplus(logits[ix,lose]-logits[ix,win]).mean()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(),1.)
        scaler.step(opt)
        scaler.update()
        if DEVICE=="cuda":
            torch.cuda.synchronize()
        previous_elapsed = elapsed
        elapsed = initial_time+time.monotonic()-start
        tokens += count
        step += 1
        if step==warm_step:
            bench_t, bench_n = time.monotonic(),tokens
        if step==warm_step+10:
            rate = (tokens-bench_n)/(time.monotonic()-bench_t)
            print("targets/s",round(rate),"projected targets",round(rate*budget),
                  "peak GPU MiB",torch.cuda.max_memory_allocated()//2**20 if DEVICE=="cuda" else 0)
        # Keep lightweight snapshots for learning curves and matched-update comparisons.
        milestone = None
        if base:
            for minute in (30,60,120,180):
                if previous_elapsed < minute*60 <= elapsed:
                    milestone = f"base-{minute}min"
        elif step in (100,1000,5000):
            milestone = f"{name}-step{step}"
        if milestone:
            snapshot = dict(model=model.state_dict(), step=step, elapsed=elapsed, tokens=tokens)
            atomic(ROOT/(milestone+".pt"),lambda p:torch.save(snapshot,p))
            print("Saved",milestone,"at",tokens,"targets")
        if elapsed-last_save >= 300 or elapsed >= budget:
            save()
            print(name,step,round(elapsed),"seconds",tokens,"targets","loss",round(loss.item(),3),flush=True)
            last_save = elapsed

## 5. Train or resume the three-hour baseline
Training time accumulates across sessions. Checkpoints are replaced every five minutes and at the end of each training phase. Resume by running from the top with the same settings. If an allocation is interrupted before the first save, restart that phase.

In [ ]:
if DEVICE != "cuda":
    raise RuntimeError("Select a Colab GPU runtime before training.")
train("base", BASE_SECONDS)

## 6. Generate automated feedback
The frozen baseline proposes candidates and Stockfish evaluates them on CPU. The first 256 positions report labeling throughput. Labels save in chunks and resume on rerun. The preference coverage tells you how many sampled positions have a strong enough engine preference to supervise branch C.

In [ ]:
def score_candidate(engine, board, move, nodes):
    # Reset hash between candidates so each receives the same fresh search budget.
    info = engine.analyse(board,chess.engine.Limit(nodes=nodes),root_moves=[move],game=object())
    return info["score"].pov(board.turn)

def label_data():
    model = load_model("base")
    base_state = checkpoint("base")
    base_key = [base_state["id"],base_state["step"]]
    del base_state
    positions = sample_rows("train",LABEL_COUNT)
    if not positions:
        raise ValueError("No training positions.")
    path = ROOT/"labels.json"
    with chess.engine.SimpleEngine.popen_uci(ENGINE) as engine:
        signature = fingerprint([base_key,positions,engine.id,LABEL_NODES])
        cache = json.loads(path.read_text()) if path.exists() else dict(signature=signature,engine=engine.id,rows=[])
        if cache["signature"]!=signature:
            raise ValueError("Engine, baseline or label settings changed. Use a new run folder.")
        rows = cache["rows"]
        started, previous = time.monotonic(),len(rows)
        for idx,ply in positions[previous:]:
            prefix = games["train"][idx][:ply]
            board = replay(prefix)
            legal = legal_ids(board)
            logits = last_logits(model,[prefix])[0].cpu()
            candidates = sorted(legal,key=lambda m: float(logits[m]),reverse=True)[:3]
            teacher = engine.play(board,chess.engine.Limit(nodes=LABEL_NODES),game=object()).move
            candidates = list(dict.fromkeys(candidates+[ids[teacher.uci()]]))
            scores = {m:score_candidate(engine,board,chess.Move.from_uci(VOCAB[m]),LABEL_NODES)
                      for m in candidates}
            best,worst = max(scores,key=scores.get),min(scores,key=scores.get)
            gap = scores[best].score(mate_score=100000)-scores[worst].score(mate_score=100000)
            rows.append([idx,ply,legal,[best,worst] if gap>=100 else None])
            if len(rows)%256==0 or len(rows)==len(positions):
                save_json(path,cache)
                print("labeled",len(rows),"positions;",round((len(rows)-previous)/(time.monotonic()-started),1),"/s",flush=True)
    if not any(r[3] for r in rows):
        raise ValueError("No clear preferences. Increase LABEL_COUNT in a new run.")
    print("Preference coverage:",sum(bool(r[3]) for r in rows),"/",len(rows))
    return rows

In [ ]:
labels = label_data()

## 7. Fine-tune the three branches
Each starts from the same final baseline and uses the same seeded sequence of training positions. A learns the recorded move, B adds legal probability mass and C also learns engine preferences. The budget is per branch.

In [ ]:
for name in ("A","B","C"):
    train(name, BRANCH_SECONDS, labels)

## 8. Evaluate the models
Reports include unmasked legality, prediction loss, free rollouts, and engine scores on positions absent from training. The paired match uses legal moves and reports context cutoffs separately. Engine and match evaluation can run in a later session.

In [ ]:
def position_key(board):
    # python-chess 1.11.2 repetition identity includes turn, castling and legal en passant.
    return board._transposition_key()

def novel_test_rows():
    keys = {r: position_key(replay(games["test"][r[0]][:r[1]])) for r in frozen["test"]}
    remaining = set(keys.values())
    for g in games["train"]:
        board = chess.Board()
        for token in g:
            remaining.discard(position_key(board))
            board.push(chess.Move.from_uci(VOCAB[token]))
        remaining.discard(position_key(board))
        if not remaining:
            break
    return {r for r,key in keys.items() if key in remaining}

def rollout(model,prefix,seed):
    rng = torch.Generator().manual_seed(seed)
    start_ply = len(prefix)
    prefix = prefix.copy()
    board = replay(prefix)
    for ply in range(40):
        if board.is_game_over(claim_draw=True):
            break
        if len(prefix)>=CTX:
            return dict(status="context",plies=ply)
        probs = last_logits(model,[prefix])[0].cpu().softmax(0)
        token = int(torch.multinomial(probs,1,generator=rng))
        if token not in legal_ids(board):
            return dict(status="illegal",plies=ply)
        prefix.append(token)
        board.push_uci(VOCAB[token])
    return dict(status="success",plies=len(board.move_stack)-start_ply)

def evaluate(name, novel):
    model = load_model(name)
    records = []
    with chess.engine.SimpleEngine.popen_uci(ENGINE) as engine:
        for split,positions in frozen.items():
            for idx,ply in positions:
                g = games[split][idx]
                prefix = g[:ply]
                board = replay(prefix)
                legal = legal_ids(board)
                logits = last_logits(model,[prefix])[0].cpu()
                chosen = max(legal,key=lambda m: float(logits[m]))
                rec = dict(split=split,game=idx,ply=ply,
                           legal=int(logits.argmax()) in legal,
                           illegal_mass=float(1-logits.softmax(0)[legal].sum()),
                           nll=float(-logits.log_softmax(0)[g[ply]]))
                if split=="test" and (idx,ply) in novel:
                    ref = engine.analyse(board,chess.engine.Limit(nodes=TEST_NODES),game=object())
                    move = chess.Move.from_uci(VOCAB[chosen])
                    best = ref["score"].pov(board.turn)
                    score = best if move==ref["pv"][0] else score_candidate(engine,board,move,TEST_NODES)
                    rec["missed_mate"] = bool(best.is_mate() and best.mate()>0 and not (score.is_mate() and score.mate()>0))
                    if not best.is_mate() and not score.is_mate():
                        rec["cp_loss"] = max(0,best.score()-score.score())
                records.append(rec)
    runs = [rollout(model,games["test"][i][:p],SEED+k) for k,(i,p) in enumerate(openings)]
    groups = {}
    for split in frozen:
        for lo,hi in [(0,19),(20,59),(60,CTX-1)]:
            subset = [r for r in records if r["split"]==split and lo<=r["ply"]<=hi]
            if subset:
                groups[f"{split}-{lo}-{hi}"] = {k:float(np.mean([r[k] for r in subset]))
                                                  for k in ("legal","illegal_mass","nll")}
    def mean_field(key):
        values = [r[key] for r in records if key in r]
        return float(np.mean(values)) if values else None
    summary = dict(groups=groups,novel_positions=len(novel),mean_cp_loss=mean_field("cp_loss"),
                   missed_mate_positions=sum(r.get("missed_mate",False) for r in records),
                   rollout_success=sum(r["status"]=="success" for r in runs),rollouts=len(runs))
    save_json(ROOT/(name+"-evaluation.json"),dict(summary=summary,positions=records,rollouts=runs))
    print(name,summary)

def duel():
    models = {k:load_model(k) for k in ("B","C")}
    results, pairs = [], []
    for idx,ply in openings:
        scores = []
        for white in ("B","C"):
            prefix = games["test"][idx][:ply]
            board = replay(prefix)
            while len(prefix)<CTX and not board.is_game_over(claim_draw=True):
                active = white if board.turn else ("C" if white=="B" else "B")
                logits = last_logits(models[active],[prefix])[0].cpu()
                token = max(legal_ids(board),key=lambda m:float(logits[m]))
                prefix.append(token)
                board.push_uci(VOCAB[token])
            outcome = board.outcome(claim_draw=True)
            score = None if not outcome else .5 if outcome.winner is None else float((white=="C")==outcome.winner)
            scores.append(score)
            results.append(dict(game=idx,ply=ply,white=white,moves=prefix[ply:],C_score=score))
        if None not in scores:
            pairs.append(np.mean(scores))
    ci = None
    if pairs:
        samples = np.random.default_rng(SEED).choice(pairs,(2000,len(pairs))).mean(axis=1)
        ci = np.quantile(samples,[.025,.975]).tolist()
    scores = [r["C_score"] for r in results]
    summary = dict(C_wins=scores.count(1.),draws=scores.count(.5),C_losses=scores.count(0.),
                   context_cutoffs=scores.count(None),complete_pairs=len(pairs),
                   paired_score=float(np.mean(pairs)) if pairs else None,paired_95pct_ci=ci)
    save_json(ROOT/"duel.json",dict(summary=summary,games=results))
    print(summary)

In [ ]:
novel = novel_test_rows()
for name in ("base","A","B","C"):
    evaluate(name, novel)

In [ ]:
duel()